In [1]:
from math import factorial, comb
import numpy as np

D = 3

def rand_ortho_pair(N):
    v, w = np.random.randn(N), np.random.randn(N)
    v /= np.linalg.norm(v)
    w /= np.linalg.norm(w)
    w -= (v@w) * v
    w /= np.linalg.norm(w)
    
    assert np.isclose(np.linalg.norm(v), 1)
    assert np.isclose(np.linalg.norm(w), 1)
    assert np.isclose(v@w, 0)
    
    return v, w

def T_matrix(D):
    T = np.zeros((D+1, D+1))
    for i in range(D+1):
        T[i, i] = 2*i + 1
    for i in range(D-1):
        T[i, i+2] = -np.sqrt((i+1)*(i+2))
        T[i+2, i] = -np.sqrt((i+1)*(i+2))
    T /= 4
    return T
T_matrix = T_matrix(D)

def hermfpmul(f, g):
    """Return $m$, such that $mh_0=fg$.

    $f$, $g$ & $m$ are sequences of coefficients
    representing Hermite function series.
    """
    r = np.zeros(f.size+g.size-1)
    for i, fi in enumerate(f):
        for j, gj in enumerate(g):
            for k in range(min(i, j)+1):
                r[i+j-2*k] += fi * gj \
                        * factorial(k)*comb(i, k)*comb(j, k) \
                        * np.sqrt(float(factorial(i+j-2*k)) \
                        / (factorial(i)*factorial(j)))
    return r

n = 2000
gs, Ts = [], []
for _ in range(n):
    v, w = rand_ortho_pair(D+1)
    rho = np.outer(v, v) + np.outer(w, w)
    T = (rho @ T_matrix).trace()
    g = hermfpmul(v, v) + hermfpmul(w, w)
    Ts.append(T)
    gs.append(g)
gs, Ts = np.array(gs), np.array(Ts)

In [2]:
X = np.column_stack([gs[:,1:], Ts]) #remove g_0 to avoid bias ruining implicit polynomial fit
X_train, X_test = X[:n//2], X[-n//2:]

In [3]:
from sklearn.preprocessing import PolynomialFeatures

#ChatGPT do your thing
class ImplicitPolynomial:
    def __init__(self, degree: int):
        self.degree = degree
        self.features = PolynomialFeatures(
            degree=degree,
            include_bias=True,
        )

    def fit(self, xyz: np.ndarray) -> "ImplicitPolynomial":
        """
        Fit p(x, y, z) = 0 to points of shape (n_samples, 7).
        """
        xyz = np.asarray(xyz, dtype=float)

        if xyz.ndim != 2 or xyz.shape[1] != 7:
            raise ValueError("xyz must have shape (n_samples, 7)")

        # Design matrix:
        # [1, x, y, z, x², xy, xz, y², yz, z², ...]
        A = self.features.fit_transform(xyz)

        # Solve min ||A c|| subject to ||c|| = 1.
        _, singular_values, Vt = np.linalg.svd(A, full_matrices=False)

        self.coef_ = Vt[-1]
        self.singular_values_ = singular_values
        return self

    def predict(self, xyz: np.ndarray) -> np.ndarray:
        """
        Evaluate p(x, y, z). Points on the fitted surface give values near zero.
        """
        A = self.features.transform(np.asarray(xyz, dtype=float))
        return A @ self.coef_

    def equation(self, precision: int = 5) -> str:
        names = self.features.get_feature_names_out(['g1', 'g2', 'g3', 'g4', 'g5', 'g6', 'T'])

        terms = [
            f"{coef:+.{precision}g}*{name}"
            for coef, name in zip(self.coef_, names)
            if abs(coef) > 10 ** (-precision)
        ]
        return " ".join(terms) + " = 0"

p = ImplicitPolynomial(3)
#errors at degrees
#1: 0.0225921552893911
#2: 3.3056851056237902e-06
#3: 2.06347681996899e-14
p

In [4]:
p.fit(X_train)

In [5]:
np.mean(p.predict(X_test))

np.float64(4.224315341971874e-14)

In [6]:
p.equation(2)

'-0.46*g1 +0.42*g3 -0.081*g5 +0.12*g1 g2 +0.03*g1 g4 -0.21*g1 g6 +0.36*g1 T -0.27*g2 g3 +0.32*g2 g5 +0.084*g3 g4 +0.16*g3 g6 -0.23*g3 T -0.21*g4 g5 +0.028*g5 g6 -0.12*g5 T -0.012*g1^2 g3 +0.034*g1 g2^2 -0.095*g1 g2 g4 +0.074*g1 g2 g6 -0.057*g1 g2 T +0.012*g1 g3^2 -0.016*g1 g3 g5 +0.054*g1 g4^2 -0.054*g1 g4 g6 -0.016*g1 g4 T +0.09*g1 g6 T -0.06*g1 T^2 +0.017*g2^2 g3 -0.043*g2^2 g5 -0.058*g2 g3 g6 +0.11*g2 g3 T +0.053*g2 g4 g5 +0.032*g2 g5 g6 -0.14*g2 g5 T -0.016*g3 g4^2 +0.039*g3 g4 g6 -0.02*g3 g4 T -0.062*g3 g6 T -0.015*g4^2 g5 -0.024*g4 g5 g6 +0.09*g4 g5 T -0.036*g5 g6 T +0.088*g5 T^2 = 0'

In [7]:
import sympy as sp

#ChatGPT do your thing
def to_sympy(model, variable_names=None, float_precision=None):
    """
    Convert a fitted ImplicitPolynomial model to a SymPy polynomial.

    Returns
    -------
    expr : sympy.Expr
        The polynomial expression p(x0, ..., x6).
    poly : sympy.Poly
        The same expression as a multivariate SymPy polynomial.
    symbols : tuple[sympy.Symbol, ...]
        The polynomial variables.
    """
    if not hasattr(model, "coef_"):
        raise ValueError("The model must be fitted first.")

    n_variables = model.features.n_features_in_

    if variable_names is None:
        variable_names = [f"x{i}" for i in range(n_variables)]

    if len(variable_names) != n_variables:
        raise ValueError(
            f"Expected {n_variables} variable names, "
            f"got {len(variable_names)}."
        )

    symbols = sp.symbols(" ".join(variable_names))

    expr = sp.Integer(0)

    for coefficient, powers in zip(
        model.coef_,
        model.features.powers_,
    ):
        if float_precision is None:
            coefficient_sympy = sp.Float(coefficient)
        else:
            coefficient_sympy = sp.Float(
                coefficient,
                float_precision,
            )

        monomial = sp.prod(
            symbol ** int(power)
            for symbol, power in zip(symbols, powers)
        )

        expr += coefficient_sympy * monomial

    expr = sp.expand(expr)
    poly = sp.Poly(expr, *symbols)

    return expr, poly, symbols

expr, poly, variables = to_sympy(
    p,
    variable_names=[
        'g1', 'g2', 'g3', 'g4', 'g5', 'g6', 'T'
    ],
)

display(poly)

Poly(0.00334425770157113*g1**3 + 2.5951463200613e-15*g1**2*g2 - 0.0116049436610791*g1**2*g3 - 4.75522399234762e-14*g1**2*g4 + 0.00854805177923662*g1**2*g5 + 5.95079541199084e-14*g1**2*g6 - 2.24924245895153e-14*g1**2*T + 1.42572585681844e-13*g1**2 + 0.034278641441252*g1*g2**2 + 3.93140381360624e-14*g1*g2*g3 - 0.094609596128256*g1*g2*g4 - 7.39269756522276e-14*g1*g2*g5 + 0.0740282999370126*g1*g2*g6 - 0.0567539351719117*g1*g2*T + 0.120602112240302*g1*g2 + 0.0117049019555228*g1*g3**2 + 6.33000596383937e-14*g1*g3*g4 - 0.0164515726210247*g1*g3*g5 - 1.02022557069148e-13*g1*g3*g6 + 3.17801340798951e-15*g1*g3*T - 2.64054403342762e-13*g1*g3 + 0.053508123225378*g1*g4**2 + 8.40300051763165e-15*g1*g4*g5 - 0.0537306111839079*g1*g4*g6 - 0.0163834498744493*g1*g4*T + 0.030036324769823*g1*g4 + 0.00535081232253277*g1*g5**2 + 2.65937445675934e-14*g1*g5*g6 + 6.82787160144471e-14*g1*g5*T + 8.03174800972917e-14*g1*g5 - 0.00535081232252243*g1*g6**2 + 0.0897358506602189*g1*g6*T - 0.206392456518563*g1*g6 - 0.060

In [8]:
#filter coefficients
poly = sp.Poly.from_dict(
    {k:v for k, v in poly.as_dict().items() if abs(float(v))>1e-10},
    *poly.gens
)
poly

Poly(0.00334425770157113*g1**3 - 0.0116049436610791*g1**2*g3 + 0.00854805177923662*g1**2*g5 + 0.034278641441252*g1*g2**2 - 0.094609596128256*g1*g2*g4 + 0.0740282999370126*g1*g2*g6 - 0.0567539351719117*g1*g2*T + 0.120602112240302*g1*g2 + 0.0117049019555228*g1*g3**2 - 0.0164515726210247*g1*g3*g5 + 0.053508123225378*g1*g4**2 - 0.0537306111839079*g1*g4*g6 - 0.0163834498744493*g1*g4*T + 0.030036324769823*g1*g4 + 0.00535081232253277*g1*g5**2 - 0.00535081232252243*g1*g6**2 + 0.0897358506602189*g1*g6*T - 0.206392456518563*g1*g6 - 0.0601966386285584*g1*T**2 + 0.361179831771432*g1*T - 0.459835433968291*g1 + 0.0170660936192991*g2**2*g3 - 0.0433508340233724*g2**2*g5 + 0.00709424189646892*g2*g3*g4 - 0.0582851447265091*g2*g3*g6 + 0.110056060802296*g2*g3*T - 0.269347727753002*g2*g3 + 0.0528773570978631*g2*g4*g5 + 0.0320514141985507*g2*g5*g6 - 0.139884347343617*g2*g5*T + 0.316898490587104*g2*g5 - 0.00341321872385158*g3**3 + 0.00732690152506814*g3**2*g5 - 0.0163834498745156*g3*g4**2 + 0.038885535286129

In [9]:
sp.Poly(poly, sp.Symbol('T'))

Poly((-0.0601966386285584*g1 + 0.0879228183010161*g5)*T**2 + (-0.0567539351719117*g1*g2 - 0.0163834498744493*g1*g4 + 0.0897358506602189*g1*g6 + 0.361179831771432*g1 + 0.110056060802296*g2*g3 - 0.139884347343617*g2*g5 - 0.020065546209552*g3*g4 - 0.0622786629630771*g3*g6 - 0.229368298243182*g3 + 0.0897358506602898*g4*g5 - 0.0360435897240054*g5*g6 - 0.117230424401527*g5)*T + 0.00334425770157113*g1**3 - 0.0116049436610791*g1**2*g3 + 0.00854805177923662*g1**2*g5 + 0.034278641441252*g1*g2**2 - 0.094609596128256*g1*g2*g4 + 0.0740282999370126*g1*g2*g6 + 0.120602112240302*g1*g2 + 0.0117049019555228*g1*g3**2 - 0.0164515726210247*g1*g3*g5 + 0.053508123225378*g1*g4**2 - 0.0537306111839079*g1*g4*g6 + 0.030036324769823*g1*g4 + 0.00535081232253277*g1*g5**2 - 0.00535081232252243*g1*g6**2 - 0.206392456518563*g1*g6 - 0.459835433968291*g1 + 0.0170660936192991*g2**2*g3 - 0.0433508340233724*g2**2*g5 + 0.00709424189646892*g2*g3*g4 - 0.0582851447265091*g2*g3*g6 - 0.269347727753002*g2*g3 + 0.0528773570978631*

In [10]:
from itertools import islice

for g, T in islice(zip(gs, Ts), 10):
    print(poly.subs({sp.Symbol('g1'): g[1],
                     sp.Symbol('g2'): g[2],
                     sp.Symbol('g3'): g[3],
                     sp.Symbol('g4'): g[4],
                     sp.Symbol('g5'): g[5],
                     sp.Symbol('g6'): g[6],
                     sp.Symbol('T'): T}))

1.12103035188049e-13
2.72712408211362e-13
5.86031223548389e-13
-1.35773337017753e-13
-2.95537574447613e-13
1.86739512741951e-13
2.25264251696444e-13
5.08051933856279e-13
-2.45171938306754e-13
1.38267002014469e-13
